## Package Import

In [1]:
#!pip install --quiet h5py einops jiwer
#!pip install --quiet https://github.com/kpu/kenlm/archive/master.zip pyctcdecode

import os
import sys,random
import re
import math
import shutil
import subprocess
import collections
import multiprocessing
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
from tqdm.auto import tqdm
import jiwer
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW

import warnings
warnings.filterwarnings("ignore",)

# --- 3. GLOBAL CONFIGURATION ---
print("Torch", torch.__version__, "CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LOGIT_TO_PHONEME = [
'BLANK',    # "BLANK" = CTC blank symbol
'AA', 'AE', 'AH', 'AO', 'AW',
'AY', 'B', 'CH', 'D', 'DH',
'EH', 'ER', 'EY', 'F', 'G',
'HH', 'IH', 'IY', 'JH', 'K',
'L', 'M', 'N', 'NG', 'OW',
'OY', 'P', 'R', 'S', 'SH',
'T', 'TH', 'UH', 'UW', 'V',
'W', 'Y', 'Z', 'ZH',
'|',    # "|" = silence token
]


C:\Users\ren11\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch 2.9.1+cu128 CUDA available: True


### Hyperparameters

In [2]:
downsample_factor = 6 #downsample timeseries by a factor
train = True
test = True

## Signal2Phoneme

### Data Loading

In [3]:
class BrainDataset(Dataset):
    def __init__(self, index_df, index_df2, cache_size=4):
        self.df = index_df
        self.df2 = index_df2
        self.df = pd.concat([self.df,self.df2])
        print(self.df.shape)
        self._cache_size = cache_size
        self._file_cache = collections.OrderedDict()
        self.all_feats = []
        self.all_tgt = []
        self.dates = []
        self.all_rate = []
        self.all_slength = []
        self.all_tlength = []
        self.all_targets = []
        
    def __len__(self):
        return len(self.df)

    def _open_file(self, path):
        if path in self._file_cache:
            self._file_cache.move_to_end(path)
            return self._file_cache[path]
        f = h5py.File(path, 'r')
        self._file_cache[path] = f
        if len(self._file_cache) > self._cache_size:
            old_path, old_f = self._file_cache.popitem(last=False)
            try: old_f.close()
            except: pass
        return f
            
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        f = self._open_file(row['h5_path'])
        g = f[row['group']]
        feats = torch.from_numpy(g['input_features'][()].astype('float32'))
        tgt = torch.from_numpy(g['seq_class_ids'][()].astype('int64'))
        date = row['date']
        return feats, tgt, date
    
    def check_len_x_len_y_rate(self):
        return self.all_rate


    def collate_for_ctc(self,batch, pad_id=0, blank_id=0, enforce_multiple=6):
        """
        batch: list of (x, y)
          x: Tensor [T_x, C]  (variable T_x)
          y: 1D LongTensor [T_y] possibly padded with pad_id (e.g., 0)
        pad_id: the value used in dataset to pad target sequences
        blank_id: CTC blank index (must NOT appear in cleaned targets)
        enforce_multiple: pad time T_x up to multiple of this (e.g., for strided convs)

        Returns:
          x_padded: [B, C, T_max]  (you can permute later if needed)
          targets_concat: 1D LongTensor of all cleaned targets concatenated
          x_lens: [B] input lengths before time padding
          t_lens: [B] cleaned target lengths (no pads, no blanks)
        """
        xs, ys,dates = zip(*batch)

        # --- input time lengths and padding to multiple ---
        x_lens = torch.tensor([x.shape[0] for x in xs], dtype=torch.long)
        T_max = int(x_lens.max().item())
        if enforce_multiple is not None and enforce_multiple > 1:
            mod = T_max % enforce_multiple
            if mod != 0:
                T_max += (enforce_multiple - mod)

        C = xs[0].shape[1]
        B = len(xs)
        x_padded = torch.zeros(B, C, T_max, dtype=torch.float32)
        for i, x in enumerate(xs):
            # x is [T, C] -> store as [B, C, T]
            x_padded[i, :, :x.shape[0]] = x.permute(1, 0)

        # --- clean targets: remove pads, ensure no blank in targets ---
        ys = torch.stack(ys)  # (B, T_max)
        mask = ys != pad_id
        t_lens = mask.sum(dim=1)  # (B,)
        targets_concat = ys[mask]
        self.all_slength.append(x_lens)
        self.all_tlength.append(t_lens)
        self.all_rate.append(x_lens/t_lens)
        self.all_targets.append(targets_concat)
        
        # final sanity checks
        if targets_concat.numel() > 0:
            assert (targets_concat != blank_id).all(), "Targets still contain blank id after cleaning."
        return x_padded, targets_concat, x_lens, t_lens, dates

def filter_dataframe_by_length(df, downsample_factor=6):
    print(f"Original dataframe size: {len(df)}")
    valid_indices,max_tgt_length, min_tgt_length, max_signal, min_signal = [],0,9999,0,9999
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Filtering samples"):
        try:
            with h5py.File(row['h5_path'], 'r') as hf:
                g = hf[row['group']]
                target_len = (g['seq_class_ids'][()]!=0).sum()
                if target_len > 1 and target_len <= g['input_features'].shape[0] // downsample_factor:
                    valid_indices.append(idx)
        except Exception:
            pass
    filtered_df = df.loc[valid_indices].reset_index(drop=True)
    
    print(f"Filtered dataframe size: {len(filtered_df)} ({len(df) - len(filtered_df)} samples removed)")
    return filtered_df

In [4]:
INPUT_DIR = Path('./t15_copyTask_neuralData/hdf5_data_final')
import sys
import re
train_rows = []
dates_dict = {}
counter = 0
for p in INPUT_DIR.rglob("data_train.hdf5"):
    try:
        with h5py.File(p, 'r') as hf:
            match = re.search(r"\d{4}\.\d{2}\.\d{2}", str(p))
            if match:
                dates_dict[match.group(0)] = counter
                counter+=1
            for g in hf.keys():
                if g.startswith('trial_'):
                    train_rows.append({'h5_path': str(p), 'group': g,'date':dates_dict[match.group(0)]})
    except (IOError, OSError):
        print(f"Warning: Could not read {p}, skipping.")
train_df_unfiltered = pd.DataFrame(train_rows)
# --- THIS IS THE FIX ---
# Change the downsample factor to 2 to match our new model architecture.
train_df_index = filter_dataframe_by_length(train_df_unfiltered, downsample_factor=downsample_factor)
# --------------------
print("\nBuilding and filtering validation file index...")
val_rows = []
for p in INPUT_DIR.rglob("data_val.hdf5"):
    try:
        with h5py.File(p, 'r') as hf:
            match = re.search(r"\d{4}\.\d{2}\.\d{2}", str(p))             
            for g in hf.keys():
                if g.startswith('trial_'):
                    val_rows.append({'h5_path': str(p), 'group': g,'date':dates_dict[match.group(0)]})
    except (IOError, OSError):
        print(f"Warning: Could not read {p}, skipping.")
val_df_unfiltered = pd.DataFrame(val_rows)
# --- THIS IS THE FIX ---
val_df_index = filter_dataframe_by_length(val_df_unfiltered, downsample_factor=downsample_factor)
# --------------------

Original dataframe size: 8072


Filtering samples: 100%|█████████████████████████████████████████████████████████| 8072/8072 [00:07<00:00, 1091.76it/s]


Filtered dataframe size: 8071 (1 samples removed)

Building and filtering validation file index...
Original dataframe size: 1426


Filtering samples: 100%|█████████████████████████████████████████████████████████| 1426/1426 [00:01<00:00, 1078.03it/s]

Filtered dataframe size: 1426 (0 samples removed)


In [5]:
if train or test: train_ds = BrainDataset(train_df_index,val_df_index)

(9497, 3)


In [6]:
np.random.seed(10)
all_idx = len(train_df_index)+len(val_df_index)
all_index = np.random.permutation(all_idx)
print(all_index,len(all_index))

[8928 2589 5824 ... 1344 7293 1289] 9497


In [7]:
if train or test:
    train_dataset = torch.utils.data.Subset(train_ds, all_index[:all_idx//10*9+100])
    test_dataset  = torch.utils.data.Subset(train_ds, all_index[all_idx//10*9+100:])

    # Set num_workers=0 to disable multiprocessing for stability in Kaggle.
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=train_ds.collate_for_ctc, num_workers=0)
    val_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=train_ds.collate_for_ctc, num_workers=0)

    print(len(train_loader), len(val_loader))
    print(f"\nDataLoaders created. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

271 27

DataLoaders created. Train batches: 271, Val batches: 27


In [8]:
def seed_everything(seed: int = 42):
    # Python built in random
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Make CuDNN deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Optional but recommended
    os.environ["PYTHONHASHSEED"] = str(seed)

# usage
seed_everything(42)

In [15]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(900.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class ConvStem(nn.Module):
    def __init__(self, in_ch, d_model):
        super().__init__()
        self.d_model = d_model
        self.in_ch = in_ch

        self.cov0 = nn.Sequential(nn.Conv1d(in_ch, d_model , kernel_size=15, stride=3, padding=7,bias=False),nn.ReLU())
        self.cov01 = nn.Sequential(nn.Conv1d(d_model , d_model , kernel_size=11, stride=2, padding=5,bias=False),nn.ReLU(),
                                  nn.Conv1d(d_model , d_model//2 , kernel_size=7, stride=1, padding=3,bias=False),nn.ReLU(),)

        self.cov1 = nn.Sequential(nn.Conv1d(in_ch, d_model , kernel_size=11, stride=3, padding=5,bias=False),nn.ReLU())
        self.cov11 = nn.Sequential(nn.Conv1d(d_model , d_model, kernel_size=7, stride=2, padding=3,bias=False),nn.ReLU(),
                                  nn.Conv1d(d_model, d_model//2 , kernel_size=5, stride=1, padding=2,bias=False),nn.ReLU(),)
        
        self.cov2 = nn.Sequential(nn.Conv1d(in_ch, d_model , kernel_size=5, stride=3, padding=2,bias=False),nn.ReLU())
        self.cov21 = nn.Sequential(nn.Conv1d(d_model , d_model//2 , kernel_size=3, stride=2, padding=1,bias=False),nn.ReLU(),)
        
    def forward(self, x): 
        emb1 = self.cov01(self.cov0(x)).permute(0, 2, 1)
        emb2 = self.cov11(self.cov1(x)).permute(0, 2, 1)
        emb3 = self.cov21(self.cov2(x)).permute(0, 2, 1)
        return emb1,emb2,emb3

class BrainToTextModel(nn.Module):
    def __init__(self, in_ch=512//2, d_model=512, nhead=4, num_layers=2, vocab_size=len(LOGIT_TO_PHONEME)):
        super().__init__()
        print(vocab_size,'pheneme used')
        scaler = 4
        day_dim = 32
        self.conv = ConvStem(in_ch+day_dim, d_model)
        self.pos_enc = PositionalEncoding(scaler*d_model//8)
        
        encoder_layer1 = nn.TransformerEncoderLayer(
            d_model=scaler*d_model//8, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, activation='relu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer1, num_layers=num_layers)
        
        self.fc = nn.Sequential(nn.Linear(scaler*d_model//8, vocab_size,bias=False))
        self.act=nn.ReLU()
        
        self.day_reminder = nn.Parameter(nn.init.xavier_uniform_(torch.empty(len(dates_dict),day_dim)))
        
    def forward(self, x, date_idx):
#         x = x[:,256:,:]
        date_ref = self.day_reminder[day_idx][:,:,None].repeat(1,1,x.shape[-1])
        mask = x!=0
        x = torch.cat([x,date_ref], 1)
        x *= mask[:,0:1,:]
        
        emb1,emb2,emb3 = self.conv(x)
        x = 0.5*emb1+0.3*emb2+0.2*emb3

        x = self.pos_enc(x)
        x = self.act(self.transformer(x))
        logits = self.fc(x)
        return F.log_softmax(logits, dim=2)


model = BrainToTextModel().to(device)
print(f"Model Initialized. Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} Million")

41 pheneme used
Model Initialized. Total parameters: 13.90 Million


In [16]:
import random
def batch_time_stretch_to_factor(
    x: torch.Tensor,
    min_rate: float = 0.80,
    max_rate: float = 1.26,
    downsample_factor: int = 6,
):
    """
    x: [B, C, T] tensor
    Returns:
        x_out: [B, C, T_pad] where T_pad is the smallest multiple of
               downsample_factor that is >= round(T * rate)
        rate:  the stretch rate used
    """
    B, C, T = x.shape

    # One stretch rate for entire batch
    rate = random.uniform(min_rate, max_rate)

    # New length after stretching
    new_len = max(1, int(round(T * rate)))

    # Stretch whole batch in one shot
    x_stretch = F.interpolate(
        x, size=new_len, mode="linear", align_corners=False
    )  # [B, C, new_len]

    # Compute padded length as multiple of downsample_factor
    if downsample_factor <= 0:
        raise ValueError("downsample_factor must be positive")

    T_pad = math.ceil(new_len / downsample_factor) * downsample_factor

    # If already a multiple, no padding
    if T_pad == new_len:
        return x_stretch, rate

    pad_right = T_pad - new_len
    # Pad along time dimension on the right: (left, right)
    x_out = F.pad(x_stretch, (0, pad_right))

    return x_out, rate


In [17]:
from check_confusion import build_phoneme_confusion_from_str_indices
import data_augmentation
from torch.cuda.amp import autocast, GradScaler

import importlib
importlib.reload(data_augmentation)
gauss_smooth, TimeSeriesAugment = data_augmentation.gauss_smooth, data_augmentation.TimeSeriesAugment

NUM_EPOCHS = 300
MAX_LR = 2e-3
WEIGHT_DECAY = 1e-6
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)
best_wer = float('inf')
augment = TimeSeriesAugment()

import matplotlib.pyplot as plt
if True:
    print(f"🚀 Starting training for {NUM_EPOCHS} epochs with OneCycleLR scheduler.")
    for epoch in range(1, NUM_EPOCHS + 1):
        optimizer = Adam(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch} [train]", leave=False)
        deleted_frames, total_loss, total_val_loss, sample_count = 0,0,0,0
        all_preds, all_refs = [],[]
        for i, (x, targets, x_lens, t_lens, dates) in enumerate(pbar):
            if targets.numel() == 0: continue
            x, targets, x_lens, t_lens, dates = x.to(device), targets.to(device), x_lens.to(device), t_lens.to(device), torch.tensor(list(dates)).to(device)
            factor = 1
            if random.random()<0.9:
                x = augment(x)
                if random.random()<0.8:
                    x, factor = batch_time_stretch_to_factor(x,downsample_factor=downsample_factor)
            x = x[:,256:,:]
            x = gauss_smooth(x.transpose(2,1),device)
            input_lengths = torch.ceil(x_lens*factor).long()//downsample_factor
            log_probs = model(x,dates)
            # --------------------
            loss = ctc_loss(log_probs.permute(1, 0, 2), targets, input_lengths, t_lens)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss+=loss.item()
            pbar.set_postfix({'loss': loss.item(), 'lr': MAX_LR})#scheduler.get_last_lr()[0]})
            
        if epoch<125:
            MAX_LR/=1.04
        elif epoch<250:
            MAX_LR/=1.02
        model.eval()
        with torch.no_grad():
            for x, targets, x_lens, t_lens, dates in tqdm(val_loader, desc=f"Epoch {epoch} [val]", leave=False):
                if targets.numel() == 0: continue
                x, targets, x_lens, t_lens, dates = x.to(device), targets.to(device), x_lens.to(device), t_lens.to(device), torch.tensor(list(dates)).to(device)
                x = x[:,256:,:]
                x = gauss_smooth(x.transpose(2,1),device)
                
                log_probs = model(x,dates) 
                input_lengths = x_lens//downsample_factor

                total_val_loss += ctc_loss(log_probs.permute(1, 0, 2), targets, input_lengths, t_lens).item()

                decoded = log_probs.argmax(-1).cpu().numpy()
                target_offset = 0
                for i, L in enumerate(t_lens.cpu().numpy()):
                    pred_indices = [p for j,p in enumerate(decoded[i][:input_lengths[i]]) if (j==0 or p!=decoded[i][j-1]) and p!=0]
                    all_preds.append(" ".join(map(str, pred_indices)))
                    all_refs.append(" ".join(map(str, targets[target_offset:target_offset+L].cpu().numpy())))
                    target_offset += L
        
        avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0
        wer = jiwer.wer(all_refs, all_preds) if len(all_refs) > 0 else 1.0
        print(f"Epoch {epoch}: Train Loss={total_loss/len(train_loader):.4f} | Val Loss={avg_val_loss:.4f} | WER={wer:.4f}")
        if wer < best_wer:
            best_wer = wer
            build_phoneme_confusion_from_str_indices(all_refs, all_preds)
            torch.save(model.state_dict(), f"./model_saving/best_model.pth")
            print(f"✅ New best model saved with WER: {best_wer:.4f}") 
        
    cf = build_phoneme_confusion_from_str_indices(all_refs, all_preds)
    print(cf)
else:
    state_dict = torch.load(
    r"model_saving\best_model_0.67.pth",
    map_location="cpu"
    )   

    model.load_state_dict(state_dict) 
    model.eval()
    print('Model loaded')
# print("\n✅ Training complete.") 

🚀 Starting training for 300 epochs with OneCycleLR scheduler.


Epoch 1: Train Loss=3.3944 | Val Loss=2.0232 | WER=0.6043
✅ New best model saved with WER: 0.6043


Epoch 2: Train Loss=1.8423 | Val Loss=1.3609 | WER=0.4177
✅ New best model saved with WER: 0.4177


Epoch 3: Train Loss=1.3902 | Val Loss=1.0686 | WER=0.3317
✅ New best model saved with WER: 0.3317


Epoch 4: Train Loss=1.1526 | Val Loss=0.9424 | WER=0.3008
✅ New best model saved with WER: 0.3008


...

Epoch 264: Train Loss=0.0723 | Val Loss=0.4235 | WER=0.0942


Epoch 265: Train Loss=0.0640 | Val Loss=0.4236 | WER=0.0945


Epoch 266: Train Loss=0.0742 | Val Loss=0.4236 | WER=0.0945


Epoch 267: Train Loss=0.0713 | Val Loss=0.4231 | WER=0.0942


KeyboardInterrupt: 